# Парная линейная регрессия спектра кукурузы
Каждая строка исходного CSV содержит длину волны и измеренное значение `Spectr`. Построим прогноз методом наименьших квадратов двумя способами и выясним, какую долю изменчивости спектра объясняет линейная зависимость. Оставьте CSV в одной папке с блокнотом и выполните ячейки сверху вниз.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv("Гиперспектр кукурузы.csv", sep=";", decimal=",", encoding="utf-8-sig")
assert list(data) == ["wavelength", "Spectr"] and np.isfinite(data.to_numpy()).all()
x = data.wavelength.to_numpy() * .001
y = data.Spectr.to_numpy()
print(f"Размер выборки: {len(y)}. Диапазон до масштабирования: {data.wavelength.min():.2f}–{data.wavelength.max():.2f}.")
data.head()

## Прямое вычисление
Для модели $\hat y=a_0+a_1 x$ наклон равен отношению суммы попарных произведений отклонений к сумме квадратов отклонений аргумента, а свободный член равен $\bar y-a_1\bar x$. Масштабирование меняет численное значение наклона, сохраняя прогнозы.

In [ ]:
design = np.column_stack([np.ones_like(x), x])
centered = x - x.mean()
slope = np.dot(centered, y - y.mean()) / np.dot(centered, centered)
analytic = np.array([y.mean() - slope * x.mean(), slope])
pred_analytic = design @ analytic
print(f"Ŷ = {analytic[0]:.8f} + {analytic[1]:.8f} · x")

## Градиентный спуск
Для средней квадратичной ошибки градиент равен $2X^T(Xa-y)/n$. Начинаем с нулевых коэффициентов. В качестве устойчивого шага выбираем величину, обратную максимальному собственному значению гессиана. Останавливаемся при норме градиента меньше $10^{-10}$ либо после 20 000 итераций.

In [ ]:
hessian = 2 * design.T @ design / len(y)
rate = 1 / np.linalg.eigvalsh(hessian).max()
weights = np.zeros(2)
errors = []
for iteration in range(20_001):
    residual = design @ weights - y
    errors.append(np.mean(residual**2))
    gradient = 2 * design.T @ residual / len(y)
    if np.linalg.norm(gradient) < 1e-10 or iteration == 20_000:
        break
    weights -= rate * gradient

pred_gradient = design @ weights
assert np.linalg.norm(gradient) < 1e-10
assert np.allclose(analytic, weights, rtol=1e-7, atol=1e-8)
assert np.max(np.diff(errors)) < 1e-13
print(f"Шаг {rate:.8f}, итераций {iteration}. Ŷ = {weights[0]:.8f} + {weights[1]:.8f} · x")

## Качество прогнозов и кривая обучения
Сравним методы по средней квадратичной ошибке MSE, средней абсолютной ошибке MAE и коэффициенту детерминации R² на исходной обучающей выборке.

In [ ]:
def scores(fitted):
    error = y - fitted
    return [np.mean(error**2), np.mean(np.abs(error)), 1 - error @ error / np.sum((y-y.mean())**2)]

summary = pd.DataFrame([scores(pred_analytic), scores(pred_gradient)],
                       index=["Аналитический МНК", "Градиентный спуск"], columns=["MSE", "MAE", "R²"])
display(summary.round(7))
sort = np.argsort(x)
fig, axs = plt.subplots(2, 2, figsize=(12, 7.5), constrained_layout=True)
for axis, pred, method in zip(axs[0], [pred_analytic, pred_gradient], summary.index):
    axis.scatter(x, y, s=11, alpha=.65, color="#335d70", label="Наблюдения")
    axis.plot(x[sort], pred[sort], color="#c6512d", lw=2.2, label="Прогноз")
    axis.set(xlabel="Длина волны × 0,001", ylabel="Spectr", title=method)
    axis.legend()
axs[1,0].plot(errors, color="#167555")
axs[1,0].axhline(summary.iloc[0,0], color="#dd893b", ls="--", label="Минимум МНК")
axs[1,0].set(xscale="symlog", yscale="log", xlabel="Итерация", ylabel="MSE", title="Сходимость")
axs[1,0].legend()
axs[1,1].plot(x[sort], (y-pred_analytic)[sort], color="#465690")
axs[1,1].axhline(0, lw=.8, color="black")
axs[1,1].set(xlabel="Длина волны × 0,001", ylabel="Измерение − прогноз", title="Остатки")
plt.show()

In [ ]:
worst = np.argmax(abs(y-pred_analytic))
print(f"Прямая объясняет {100*summary.iloc[0,2]:.1f}% дисперсии Spectr. Средний модуль ошибки {summary.iloc[0,1]:.4f}.")
print(f"Максимальный по модулю остаток {y[worst]-pred_analytic[worst]:+.4f} при длине волны {data.wavelength.iloc[worst]:.2f}.")
print(f"Максимальная разница между прогнозами двух моделей: {max(abs(pred_analytic-pred_gradient)):.3g}.")

Оба способа находят один минимум MSE. На графике остатков видны интервалы, где серия соседних измерений лежит по одну сторону прямой. Линейная модель передаёт общий тренд спектра и сглаживает его локальные пики и спады. Результаты на новой партии кукурузы потребовали бы отдельной проверки.